# Research Experiment: Does Prompt Linguistics Predict LLM Output Quality?

**Hypothesis 1**: Prompts with a higher noun/verb ratio, lower pronoun density, and greater semantic specificity produce
lower-entropy, higher-quality LLM outputs.

## What we test

| Feature | What it measures |
|---|---|
| **LSS** (Linguistic Specificity Score) | Noun+verb density vs pronoun+function word density |
| **Prompt Surprisal** | How "unexpected" each token is to GPT-2 (proxy for model familiarity) |
| **Semantic Density** | How tightly clustered the content words are in embedding space |
| **Pronoun Ratio** | % of tokens that are pronouns (ambiguity proxy) |
| **Named Entity Count** | Hard anchors that collapse probability distributions |

## Experiment design

We take **5 topic pairs** — each pair has a *vague* and a *specific* version of the same question.
We extract all features from each prompt, then (optionally) send both to an LLM and score the outputs.
Finally we correlate features → quality.

---
> **Run order**: execute cells top to bottom. Cell 2 installs all dependencies.  
> **LLM calls**: set `RUN_LLM = True` in the config cell and ensure your API key is in the environment.

In [1]:
# ── CELL 1: Install all dependencies ──────────────────────────────────────────
# Run this once. Restart the kernel after this cell completes.
#
# FIX: NumPy 2.x removed numpy.core.multiarray which PyTorch C-extensions
# still import. We pin to numpy<2.0 (1.26.x) which is compatible with
# torch, transformers, spaCy, scipy, and matplotlib all at once.

import subprocess, sys

# Step 1: pin numpy FIRST before anything else installs a newer version
subprocess.run(
    [sys.executable, "-m", "pip", "install", "numpy<2.0", "-q", "--force-reinstall"],
    check=False
)

packages = [
    "spacy",
    "transformers",
    "torch",          # CPU-only is fine for GPT-2 perplexity
    "matplotlib",
    "seaborn",
    "scipy",
    "pandas",
    "scikit-learn",
    "litellm",        # LLM routing (already in project)
    "python-dotenv",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

# Download spaCy English model (medium — has word vectors for semantic density)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_md", "-q"], check=False)

# Verify numpy version is correct
import numpy as np
major = int(np.__version__.split(".")[0])
if major >= 2:
    print(f"WARNING: numpy {np.__version__} still active — restart the kernel now.")
else:
    print(f"numpy {np.__version__} OK (< 2.0)")

print("All dependencies installed. RESTART THE KERNEL before running Cell 2.")

All dependencies installed.


In [2]:
# ── CELL 2: Imports ────────────────────────────────────────────────────────────

import os, re, math, warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

import spacy
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv(dotenv_path=Path("..") / ".env")  # load API keys from project root

# ── Load spaCy model
nlp = spacy.load("en_core_web_md")

# ── Load GPT-2 (for surprisal / perplexity scoring)
print("Loading GPT-2 for surprisal scoring...")
gpt2_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2_model     = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_model.eval()

sns.set_theme(style="whitegrid", palette="muted")
print("Ready.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\dpokh\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\dpokh\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "c:\Users\dpokh\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "c:\Users\dpokh\anaconda3\Lib\site-packa

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\dpokh\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\dpokh\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "c:\Users\dpokh\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "c:\Users\dpokh\anaconda3\Lib\site-packa

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\dpokh\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\dpokh\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "c:\Users\dpokh\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "c:\Users\dpokh\anaconda3\Lib\site-packa

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [ ]:
# ── CELL 3: Configuration ─────────────────────────────────────────────────────

# Set to True to send prompts to an LLM and score outputs
RUN_LLM = True

# Which model to use for evaluation (litellm routing)
LLM_MODEL = "gpt-4o-mini"     # cheap, fast; change to "claude-3-5-haiku-20241022" etc.

# Temperature for LLM evaluation calls
LLM_TEMPERATURE = 0.2

print(f"RUN_LLM = {RUN_LLM}")
print(f"LLM_MODEL = {LLM_MODEL}")
if not RUN_LLM:
    print("[INFO] LLM calls disabled. Will analyse prompts only. Set RUN_LLM=True to enable LLM scoring.")

---
## Part A — The Feature Extraction Engine

Three feature modules:

1. **POS Analyser** — spaCy POS tags → LSS, pronoun ratio, NE count
2. **Surprisal Scorer** — GPT-2 → per-token surprisal → mean surprisal, prompt perplexity
3. **Semantic Density Scorer** — spaCy word vectors → average cosine similarity among content words

In [ ]:
# ── CELL 4: POS Analyser ───────────────────────────────────────────────────────
#
# LSS = (NN + NNP + NNPS + NNS + VB + VBD + VBG + VBN + VBP + VBZ)
#      / (PRP + PRP$ + DT + IN + CC + UH + 1)   ← +1 avoids div/0
#
# Higher LSS = more content words relative to function/ambiguity words

NOUN_TAGS  = {"NN", "NNS", "NNP", "NNPS"}
VERB_TAGS  = {"VB", "VBD", "VBG", "VBN", "VBP", "VBZ"}
PRONOUN_TAGS = {"PRP", "PRP$"}
FUNCTION_TAGS = {"DT", "IN", "CC", "UH", "EX", "RP"}

@dataclass
class POSProfile:
    text: str
    token_count: int = 0
    noun_count: int = 0
    verb_count: int = 0
    pronoun_count: int = 0
    function_count: int = 0
    ne_count: int = 0
    lss: float = 0.0
    pronoun_ratio: float = 0.0
    content_ratio: float = 0.0
    pos_dist: dict = field(default_factory=dict)

def analyse_pos(text: str) -> POSProfile:
    doc = nlp(text)
    profile = POSProfile(text=text)
    pos_counts: dict[str, int] = {}

    for token in doc:
        if token.is_space:
            continue
        profile.token_count += 1
        tag = token.tag_
        pos_counts[tag] = pos_counts.get(tag, 0) + 1

        if tag in NOUN_TAGS:      profile.noun_count += 1
        if tag in VERB_TAGS:      profile.verb_count += 1
        if tag in PRONOUN_TAGS:   profile.pronoun_count += 1
        if tag in FUNCTION_TAGS:  profile.function_count += 1

    profile.ne_count = len(list(doc.ents))
    numerator   = profile.noun_count + profile.verb_count
    denominator = profile.pronoun_count + profile.function_count + 1
    profile.lss = round(numerator / denominator, 4)
    profile.pronoun_ratio = round(profile.pronoun_count / max(profile.token_count, 1), 4)
    profile.content_ratio = round(numerator / max(profile.token_count, 1), 4)
    profile.pos_dist = pos_counts
    return profile

# Quick smoke test
test = analyse_pos("The mitochondria produce ATP through oxidative phosphorylation.")
print(f"LSS: {test.lss}  |  Nouns: {test.noun_count}  |  NEs: {test.ne_count}")

In [ ]:
# ── CELL 5: Surprisal Scorer (GPT-2) ──────────────────────────────────────────
#
# Surprisal(token_t) = -log2 P(t | t_1 ... t_{n-1})
#
# High surprisal = model finds this token unexpected given what came before.
# Mean surprisal over prompt ≈ prompt entropy from model's perspective.
# Perplexity = 2^(mean_surprisal)  — lower = model "understands" the prompt better.

@dataclass
class SurprisalProfile:
    text: str
    token_surprisals: list[float] = field(default_factory=list)
    mean_surprisal: float = 0.0
    max_surprisal: float = 0.0
    perplexity: float = 0.0
    high_surprisal_tokens: list[str] = field(default_factory=list)  # top-5 most surprising

def score_surprisal(text: str, top_n: int = 5) -> SurprisalProfile:
    profile = SurprisalProfile(text=text)
    encodings = gpt2_tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids  # shape: (1, seq_len)

    with torch.no_grad():
        outputs = gpt2_model(input_ids, labels=input_ids)
        # Compute per-token log-probabilities
        logits = outputs.logits  # (1, seq_len, vocab_size)

    # Shift: predict token i from tokens 0..i-1
    shift_logits = logits[0, :-1, :]          # (seq_len-1, vocab)
    shift_labels = input_ids[0, 1:]           # (seq_len-1,)

    log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs[range(len(shift_labels)), shift_labels]  # (seq_len-1,)
    surprisals = (-token_log_probs / math.log(2)).tolist()  # convert to bits (log2)

    profile.token_surprisals = surprisals
    profile.mean_surprisal   = float(np.mean(surprisals)) if surprisals else 0.0
    profile.max_surprisal    = float(np.max(surprisals))  if surprisals else 0.0
    profile.perplexity       = float(2 ** profile.mean_surprisal)

    # Find highest-surprisal tokens (most unexpected)
    tokens = [gpt2_tokenizer.decode([t]) for t in shift_labels.tolist()]
    sorted_pairs = sorted(zip(surprisals, tokens), reverse=True)
    profile.high_surprisal_tokens = [tok.strip() for _, tok in sorted_pairs[:top_n]]

    return profile

# Quick smoke test
sp = score_surprisal("The mitochondria produce ATP through oxidative phosphorylation.")
print(f"Perplexity: {sp.perplexity:.2f}  |  Mean surprisal: {sp.mean_surprisal:.2f} bits")
print(f"Most surprising tokens: {sp.high_surprisal_tokens}")

In [ ]:
# ── CELL 6: Semantic Density Scorer ───────────────────────────────────────────
#
# Semantic Density = average pairwise cosine similarity between content-word vectors.
# High density → words cluster tightly in embedding space → narrow, specific topic.
# Low density  → words spread across embedding space → vague, multi-topic.
#
# We only use tokens that: (a) have a vector, (b) are not stop words, (c) are content POS.

CONTENT_POS = {"NOUN", "VERB", "ADJ", "ADV", "PROPN"}

@dataclass
class SemanticProfile:
    text: str
    content_words: list[str] = field(default_factory=list)
    semantic_density: float = 0.0   # avg pairwise cosine sim
    centroid_spread: float  = 0.0   # std dev of distances from centroid (lower = tighter)

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

def score_semantic_density(text: str) -> SemanticProfile:
    profile = SemanticProfile(text=text)
    doc = nlp(text)

    vecs = []
    words = []
    for token in doc:
        if token.pos_ in CONTENT_POS and token.has_vector and not token.is_stop:
            vecs.append(token.vector)
            words.append(token.text)

    profile.content_words = words

    if len(vecs) < 2:
        return profile  # not enough content words

    vecs_arr = np.array(vecs)

    # Average pairwise cosine similarity
    sims = []
    for i in range(len(vecs_arr)):
        for j in range(i + 1, len(vecs_arr)):
            sims.append(cosine_sim(vecs_arr[i], vecs_arr[j]))
    profile.semantic_density = round(float(np.mean(sims)), 4)

    # Centroid spread
    centroid = vecs_arr.mean(axis=0)
    dists = [cosine_sim(v, centroid) for v in vecs_arr]
    profile.centroid_spread = round(float(np.std(dists)), 4)

    return profile

# Quick smoke test
sem = score_semantic_density("The mitochondria produce ATP through oxidative phosphorylation.")
print(f"Semantic density: {sem.semantic_density}  |  Centroid spread: {sem.centroid_spread}")
print(f"Content words: {sem.content_words}")

In [ ]:
# ── CELL 7: Combined Prompt Feature Extractor ─────────────────────────────────

@dataclass
class PromptFeatures:
    label: str
    prompt: str
    topic: str
    variant: str          # "vague" | "specific"
    # POS
    lss: float = 0.0
    pronoun_ratio: float = 0.0
    content_ratio: float = 0.0
    ne_count: int = 0
    noun_count: int = 0
    token_count: int = 0
    # Surprisal
    perplexity: float = 0.0
    mean_surprisal: float = 0.0
    # Semantic
    semantic_density: float = 0.0
    centroid_spread: float = 0.0
    # LLM output (filled later if RUN_LLM=True)
    llm_output: str = ""
    output_ne_count: int = 0
    output_token_count: int = 0
    output_lss: float = 0.0

def extract_features(label: str, prompt: str, topic: str, variant: str) -> PromptFeatures:
    pf = PromptFeatures(label=label, prompt=prompt, topic=topic, variant=variant)

    pos  = analyse_pos(prompt)
    surp = score_surprisal(prompt)
    sem  = score_semantic_density(prompt)

    pf.lss             = pos.lss
    pf.pronoun_ratio   = pos.pronoun_ratio
    pf.content_ratio   = pos.content_ratio
    pf.ne_count        = pos.ne_count
    pf.noun_count      = pos.noun_count
    pf.token_count     = pos.token_count
    pf.perplexity      = surp.perplexity
    pf.mean_surprisal  = surp.mean_surprisal
    pf.semantic_density = sem.semantic_density
    pf.centroid_spread  = sem.centroid_spread

    return pf

print("Feature extractor ready.")

---
## Part B — The Prompt Dataset

5 topic pairs. Each pair = one **vague** prompt + one **specific** prompt asking the *same thing*.

The vague versions have:
- High pronoun usage (`it`, `they`, `this`)
- Abstract/general nouns
- Missing semantic roles (no agent, time, location)
- Register inconsistency

The specific versions have:
- Named entities
- Domain-specific hyponyms
- All semantic roles filled
- Consistent register

In [ ]:
# ── CELL 8: Prompt Pair Dataset ────────────────────────────────────────────────

PROMPT_PAIRS = [
    {
        "topic": "Neuroscience",
        "vague":    "Can you tell me about what it does and how it affects stuff in the body and brain?",
        "specific": "Explain the mechanism by which BDNF (brain-derived neurotrophic factor) promotes synaptic plasticity in CA1 pyramidal neurons of the hippocampus.",
    },
    {
        "topic": "Concurrency Bug",
        "vague":    "I need help with something that's not working right in my code when two things run at the same time.",
        "specific": "Identify the race condition that occurs in Python asyncio when two coroutines concurrently modify a shared dictionary without asyncio.Lock, and explain how to fix it.",
    },
    {
        "topic": "History",
        "vague":    "What happened with that agreement after the big war ended and why did it cause so many problems?",
        "specific": "What were the primary economic consequences of the Treaty of Versailles (1919) on Weimar Germany's industrial output and currency stability between 1919 and 1923?",
    },
    {
        "topic": "Biochemistry",
        "vague":    "Why does the energy thing happen during that process in cells when they use the molecule?",
        "specific": "Why does ATP hydrolysis (ATP → ADP + Pᵢ) release approximately 30.5 kJ/mol under standard physiological conditions, and what makes this reaction thermodynamically favorable?",
    },
    {
        "topic": "Algorithm Optimisation",
        "vague":    "How do I make the thing faster when it has to go through lots of data and do comparisons?",
        "specific": "How do I reduce the O(N²) time complexity of pairwise Euclidean distance computation on an N×D NumPy array to O(N log N) or better, and what are the trade-offs?",
    },
]

print(f"{len(PROMPT_PAIRS)} topic pairs loaded.")
for p in PROMPT_PAIRS:
    print(f"  ▸ {p['topic']}")

---
## Part C — Feature Extraction Run

Run the full pipeline on all 10 prompts (5 vague + 5 specific).

In [ ]:
# ── CELL 9: Extract features for all prompts ───────────────────────────────────

all_features: list[PromptFeatures] = []

for pair in PROMPT_PAIRS:
    topic = pair["topic"]

    for variant, prompt in [("vague", pair["vague"]), ("specific", pair["specific"])]:
        label = f"{topic} [{variant}]"
        print(f"Processing: {label}")
        pf = extract_features(label=label, prompt=prompt, topic=topic, variant=variant)
        all_features.append(pf)
        print(f"  LSS={pf.lss:.3f}  PPL={pf.perplexity:.1f}  SemDens={pf.semantic_density:.3f}  NE={pf.ne_count}")

print(f"\nExtracted features for {len(all_features)} prompts.")

In [ ]:
# ── CELL 10: Build DataFrame ───────────────────────────────────────────────────

df = pd.DataFrame([
    {
        "label":            pf.label,
        "topic":            pf.topic,
        "variant":          pf.variant,
        "prompt":           pf.prompt[:80] + "...",
        "token_count":      pf.token_count,
        "noun_count":       pf.noun_count,
        "ne_count":         pf.ne_count,
        "pronoun_ratio":    pf.pronoun_ratio,
        "content_ratio":    pf.content_ratio,
        "lss":              pf.lss,
        "perplexity":       pf.perplexity,
        "mean_surprisal":   pf.mean_surprisal,
        "semantic_density": pf.semantic_density,
        "centroid_spread":  pf.centroid_spread,
    }
    for pf in all_features
])

# Add a numeric quality label: 1 = specific (hypothesis: better), 0 = vague
df["is_specific"] = (df["variant"] == "specific").astype(int)

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_colwidth", 50)
df[["label", "lss", "pronoun_ratio", "ne_count", "perplexity", "semantic_density"]]

---
## Part D — Visualisations

### 4.1  Per-feature comparison: vague vs specific
### 4.2  Surprisal heatmap (per-token)
### 4.3  Correlation matrix of all features
### 4.4  Scatter: LSS vs Perplexity

In [ ]:
# ── CELL 11: Figure 1 — Side-by-side bar comparison ───────────────────────────

features_to_plot = [
    ("lss",              "Linguistic Specificity Score (LSS)\n(higher = more specific)"),
    ("pronoun_ratio",    "Pronoun Ratio\n(lower = less ambiguous)"),
    ("ne_count",         "Named Entity Count\n(higher = harder anchors)"),
    ("perplexity",       "GPT-2 Perplexity\n(lower = model familiar with prompt)"),
    ("semantic_density", "Semantic Density\n(higher = tighter topic cluster)"),
]

topics = df["topic"].unique()
x = np.arange(len(topics))
width = 0.35

fig, axes = plt.subplots(len(features_to_plot), 1, figsize=(12, 4 * len(features_to_plot)))
fig.suptitle("Prompt Feature Comparison: Vague vs Specific", fontsize=15, y=1.01, fontweight="bold")

vague_df    = df[df["variant"] == "vague"].set_index("topic")
specific_df = df[df["variant"] == "specific"].set_index("topic")

for ax, (feat, title) in zip(axes, features_to_plot):
    vague_vals    = [vague_df.loc[t, feat]    for t in topics]
    specific_vals = [specific_df.loc[t, feat] for t in topics]

    bars_v = ax.bar(x - width/2, vague_vals,    width, label="Vague",    color="#e07070", alpha=0.85)
    bars_s = ax.bar(x + width/2, specific_vals, width, label="Specific", color="#5b9bd5", alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(topics, rotation=20, ha="right")
    ax.set_title(title, fontsize=11)
    ax.legend(loc="upper right")

    # Annotate bars
    for bar in bars_v:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005 * ax.get_ylim()[1],
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8, color="#c0392b")
    for bar in bars_s:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005 * ax.get_ylim()[1],
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8, color="#1a5276")

plt.tight_layout()
plt.savefig("feature_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: feature_comparison.png")

In [ ]:
# ── CELL 12: Figure 2 — Per-token surprisal heatmap ───────────────────────────
#
# Shows WHERE in each prompt the model is most uncertain (high surprisal).
# Useful to see: do vague prompts have systematic high-surprisal regions?

def get_token_surprisal_pairs(text: str):
    """Returns list of (token_text, surprisal_bits) tuples."""
    encodings = gpt2_tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids
    with torch.no_grad():
        logits = gpt2_model(input_ids).logits
    shift_logits = logits[0, :-1, :]
    shift_labels = input_ids[0, 1:]
    log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs[range(len(shift_labels)), shift_labels]
    surprisals = (-token_log_probs / math.log(2)).tolist()
    tokens = [gpt2_tokenizer.decode([t]) for t in shift_labels.tolist()]
    return list(zip(tokens, surprisals))

# Pick one pair to visualise in detail
VISUALISE_TOPIC = "Neuroscience"
pair = next(p for p in PROMPT_PAIRS if p["topic"] == VISUALISE_TOPIC)

fig, axes = plt.subplots(2, 1, figsize=(14, 5))
fig.suptitle(f"Per-Token Surprisal (GPT-2): {VISUALISE_TOPIC}", fontsize=13, fontweight="bold")

for ax, (variant, prompt) in zip(axes, [("vague", pair["vague"]), ("specific", pair["specific"])]):
    pairs = get_token_surprisal_pairs(prompt)
    tokens, surprisals = zip(*pairs) if pairs else ([], [])
    colors = ["#c0392b" if s > 12 else "#e67e22" if s > 8 else "#27ae60" for s in surprisals]
    ax.bar(range(len(surprisals)), surprisals, color=colors, alpha=0.8)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=60, ha="right", fontsize=7)
    ax.axhline(y=float(np.mean(surprisals)), color="navy", linestyle="--", linewidth=1,
               label=f"Mean: {np.mean(surprisals):.1f} bits")
    ax.set_ylabel("Surprisal (bits)")
    ax.set_title(f"[{variant.upper()}]  PPL={2**np.mean(surprisals):.1f}")
    ax.legend()

plt.tight_layout()
plt.savefig("surprisal_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: surprisal_heatmap.png")

In [ ]:
# ── CELL 13: Figure 3 — Correlation matrix ────────────────────────────────────

numeric_cols = ["lss", "pronoun_ratio", "content_ratio", "ne_count",
                "perplexity", "mean_surprisal", "semantic_density",
                "centroid_spread", "is_specific"]

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="RdYlGn", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    annot_kws={"size": 9}
)
ax.set_title("Feature Correlation Matrix\n(is_specific = 1 if prompt is specific)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nCorrelations with is_specific:")
print(corr["is_specific"].drop("is_specific").sort_values(ascending=False).to_string())

In [ ]:
# ── CELL 14: Figure 4 — LSS vs Perplexity scatter ─────────────────────────────

fig, ax = plt.subplots(figsize=(8, 6))

colors = {"vague": "#e07070", "specific": "#5b9bd5"}
markers = {"vague": "o", "specific": "s"}

for _, row in df.iterrows():
    ax.scatter(
        row["lss"], row["perplexity"],
        color=colors[row["variant"]], marker=markers[row["variant"]],
        s=120, alpha=0.85, zorder=3
    )
    ax.annotate(
        row["topic"].split()[0],
        (row["lss"], row["perplexity"]),
        textcoords="offset points", xytext=(5, 4), fontsize=7.5, alpha=0.75
    )

# Regression line
slope, intercept, r, p, se = stats.linregress(df["lss"], df["perplexity"])
x_line = np.linspace(df["lss"].min(), df["lss"].max(), 50)
ax.plot(x_line, slope * x_line + intercept, "--", color="gray", alpha=0.6,
        label=f"r={r:.2f}, p={p:.3f}")

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#e07070", markersize=10, label="Vague"),
    Line2D([0], [0], marker="s", color="w", markerfacecolor="#5b9bd5", markersize=10, label="Specific"),
    Line2D([0], [0], linestyle="--", color="gray", label=f"r={r:.2f}, p={p:.3f}"),
]
ax.legend(handles=legend_elements)

ax.set_xlabel("LSS (Linguistic Specificity Score)", fontsize=11)
ax.set_ylabel("GPT-2 Perplexity (lower = model familiar)", fontsize=11)
ax.set_title("LSS vs Prompt Perplexity", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("lss_vs_perplexity.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Pearson r = {r:.3f}, p = {p:.4f}")

In [ ]:
# ── CELL 15: Figure 5 — Semantic density spider chart ─────────────────────────

features = ["lss", "content_ratio", "semantic_density"]
labels_nice = ["LSS", "Content Ratio", "Semantic Density"]
N = len(features)

# Normalise each feature 0-1 across the dataset
df_norm = df.copy()
for f in features:
    fmin, fmax = df[f].min(), df[f].max()
    df_norm[f] = (df[f] - fmin) / (fmax - fmin + 1e-9)

angles = [n / float(N) * 2 * math.pi for n in range(N)]
angles += angles[:1]  # close the loop

fig, axes = plt.subplots(1, len(topics), figsize=(4 * len(topics), 4),
                         subplot_kw=dict(polar=True))
fig.suptitle("Prompt Quality Radar: Vague vs Specific (normalised)", fontsize=13, fontweight="bold")

for ax, topic in zip(axes, topics):
    for variant, color in [("vague", "#e07070"), ("specific", "#5b9bd5")]:
        row = df_norm[(df_norm["topic"] == topic) & (df_norm["variant"] == variant)].iloc[0]
        vals = [float(row[f]) for f in features]
        vals += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=variant)
        ax.fill(angles, vals, color=color, alpha=0.2)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels_nice, fontsize=8)
    ax.set_title(topic, fontsize=10, pad=12)
    ax.set_ylim(0, 1)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=7)

plt.tight_layout()
plt.savefig("radar_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: radar_chart.png")

---
## Part E — LLM Evaluation (RUN_LLM = True)

When enabled, both variants of each prompt are sent to the LLM.
Outputs are scored on three proxy metrics:

| Metric | What it captures |
|---|---|
| **Output LSS** | How specific the answer is (mirrors prompt quality) |
| **Output NE count** | How many named entities appear in the answer |
| **Output token count** | More specific prompts → more substantive answers |

In [ ]:
# ── CELL 16: LLM calls (optional) ─────────────────────────────────────────────

if RUN_LLM:
    import litellm
    litellm.set_verbose = False

    for pf in all_features:
        print(f"  Calling LLM: {pf.label}")
        try:
            response = litellm.completion(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": pf.prompt}],
                temperature=LLM_TEMPERATURE,
                max_tokens=400,
            )
            pf.llm_output = response.choices[0].message.content or ""

            # Score the output
            out_pos  = analyse_pos(pf.llm_output)
            pf.output_lss         = out_pos.lss
            pf.output_ne_count    = out_pos.ne_count
            pf.output_token_count = out_pos.token_count

            print(f"    Output LSS={pf.output_lss:.2f}  NE={pf.output_ne_count}  tokens={pf.output_token_count}")

        except Exception as e:
            print(f"    ERROR: {e}")

    # Add output features to df
    for pf in all_features:
        idx = df[df["label"] == pf.label].index
        df.loc[idx, "output_lss"]         = pf.output_lss
        df.loc[idx, "output_ne_count"]    = pf.output_ne_count
        df.loc[idx, "output_token_count"] = pf.output_token_count

    print("\nLLM evaluation complete.")
else:
    print("[SKIPPED] Set RUN_LLM=True in Cell 3 to run LLM evaluation.")

In [ ]:
# ── CELL 17: Figure 6 — Prompt LSS vs Output Quality (if LLM ran) ─────────────

if RUN_LLM and "output_lss" in df.columns and df["output_lss"].notna().any():

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("Prompt LSS vs LLM Output Quality Metrics", fontsize=13, fontweight="bold")

    output_metrics = [
        ("output_lss",         "Output LSS",          "How specific the answer is"),
        ("output_ne_count",    "Output Named Entities", "Factual anchors in answer"),
        ("output_token_count", "Output Token Count",   "Answer substantiveness"),
    ]

    for ax, (metric, ylabel, desc) in zip(axes, output_metrics):
        for variant, color, marker in [("vague", "#e07070", "o"), ("specific", "#5b9bd5", "s")]:
            subset = df[df["variant"] == variant]
            ax.scatter(subset["lss"], subset[metric],
                       color=color, marker=marker, s=100, label=variant, alpha=0.85)

        slope, intercept, r, p, _ = stats.linregress(df["lss"], df[metric])
        x_line = np.linspace(df["lss"].min(), df["lss"].max(), 50)
        ax.plot(x_line, slope * x_line + intercept, "--", color="gray",
                label=f"r={r:.2f} p={p:.3f}")

        ax.set_xlabel("Prompt LSS")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{ylabel}\n({desc})", fontsize=10)
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig("prompt_vs_output_quality.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("[SKIPPED] LLM output quality plot requires RUN_LLM=True.")

---
## Part F — The Prompt Quality Index (PQI)

A single composite score combining all features. In the absence of LLM labels,
we use **PCA direction 1** (the direction of maximum variance) as a quality proxy.

When LLM labels are available, weights are fitted via logistic regression on `is_specific`.

In [ ]:
# ── CELL 18: PQI — Compute and rank prompts ────────────────────────────────────

from sklearn.preprocessing import StandardScaler

PQI_FEATURES = ["lss", "content_ratio", "ne_count", "semantic_density"]
# Note: perplexity is INVERTED (lower is better), so we negate it
PQI_INVERTED = ["perplexity", "pronoun_ratio", "centroid_spread"]

df_pqi = df.copy()
for col in PQI_INVERTED:
    df_pqi[f"neg_{col}"] = -df_pqi[col]

final_feats = PQI_FEATURES + [f"neg_{c}" for c in PQI_INVERTED]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pqi[final_feats])

# Equal weights for now (PQI v1 = mean of z-scores)
df_pqi["pqi"] = X_scaled.mean(axis=1)

print("Prompt Quality Index (PQI) scores:")
print(df_pqi[["label", "variant", "pqi"]].sort_values("pqi", ascending=False).to_string(index=False))

In [ ]:
# ── CELL 19: Figure 7 — PQI bar chart ─────────────────────────────────────────

df_sorted = df_pqi.sort_values("pqi", ascending=True)
bar_colors = ["#5b9bd5" if v == "specific" else "#e07070" for v in df_sorted["variant"]]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(df_sorted["label"], df_sorted["pqi"], color=bar_colors, alpha=0.85)
ax.axvline(0, color="black", linewidth=0.8, linestyle="-")

for bar, val in zip(bars, df_sorted["pqi"]):
    ax.text(val + (0.02 if val >= 0 else -0.02), bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", ha="left" if val >= 0 else "right", fontsize=9)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#5b9bd5", label="Specific"),
    Patch(facecolor="#e07070", label="Vague"),
], loc="lower right")

ax.set_xlabel("Prompt Quality Index (PQI) — higher is better", fontsize=11)
ax.set_title("PQI Score: All Prompts Ranked", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("pqi_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Part G — Statistical Summary

Do the vague and specific prompts differ *significantly* on each feature?
We use a paired Wilcoxon signed-rank test (non-parametric, n=5 pairs).

In [ ]:
# ── CELL 20: Wilcoxon signed-rank test ────────────────────────────────────────

from scipy.stats import wilcoxon

test_features = ["lss", "pronoun_ratio", "ne_count", "perplexity", "semantic_density", "pqi"]
vague_rows    = df_pqi[df_pqi["variant"] == "vague"].sort_values("topic").reset_index(drop=True)
specific_rows = df_pqi[df_pqi["variant"] == "specific"].sort_values("topic").reset_index(drop=True)

print(f"{'Feature':<22} {'Vague mean':>12} {'Specific mean':>14} {'Direction':>12} {'W stat':>8} {'p-value':>10}")
print("-" * 82)

results = []
for feat in test_features:
    v_vals = vague_rows[feat].values
    s_vals = specific_rows[feat].values
    diff = s_vals - v_vals

    # Expected direction
    expected_higher_for_specific = feat in {"lss", "ne_count", "semantic_density", "pqi", "content_ratio"}

    if len(set(diff)) > 1:  # wilcoxon needs non-zero differences
        try:
            stat, p = wilcoxon(s_vals, v_vals, alternative="greater" if expected_higher_for_specific else "less")
        except Exception:
            stat, p = float("nan"), float("nan")
    else:
        stat, p = float("nan"), float("nan")

    direction = "specific↑" if expected_higher_for_specific else "specific↓"
    sig = "*" if p < 0.05 else "(ns)"
    print(f"{feat:<22} {v_vals.mean():>12.3f} {s_vals.mean():>14.3f} {direction:>12} {stat:>8.1f} {p:>10.4f} {sig}")
    results.append({"feature": feat, "vague_mean": v_vals.mean(), "specific_mean": s_vals.mean(),
                    "stat": stat, "p": p, "sig": sig})

print("\n* = p < 0.05  (ns) = not significant at 0.05 threshold")
print("Note: with n=5 pairs, power is low — treat as directional evidence only.")

In [ ]:
# ── CELL 21: Figure 8 — Summary dashboard ─────────────────────────────────────

fig = plt.figure(figsize=(14, 9))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle("Experiment Summary Dashboard", fontsize=14, fontweight="bold", y=1.01)

summary_features = [
    ("lss",              "LSS",              True),
    ("pronoun_ratio",    "Pronoun Ratio",     False),
    ("ne_count",         "Named Entities",   True),
    ("perplexity",       "GPT-2 Perplexity", False),
    ("semantic_density", "Semantic Density", True),
    ("pqi",              "PQI (composite)",  True),
]

for idx, (feat, title, higher_better) in enumerate(summary_features):
    ax = fig.add_subplot(gs[idx // 3, idx % 3])
    v_vals = vague_rows[feat].values
    s_vals = specific_rows[feat].values

    ax.boxplot([v_vals, s_vals], labels=["Vague", "Specific"],
               patch_artist=True,
               boxprops=dict(facecolor="#e0e0e0"),
               medianprops=dict(color="black", linewidth=2))

    for i, (vals, color) in enumerate([(v_vals, "#e07070"), (s_vals, "#5b9bd5")]):
        ax.scatter(np.random.normal(i + 1, 0.04, len(vals)), vals,
                   color=color, s=50, zorder=3, alpha=0.9)

    arrow = "↑ better" if higher_better else "↓ better"
    ax.set_title(f"{title}\n({arrow})", fontsize=9)
    ax.tick_params(labelsize=8)

plt.savefig("summary_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: summary_dashboard.png")

---
## Part H — Prompt Rewriter (Proof of Concept)

The simplest form of "prompt improvement" based on what we've measured:

1. **Replace pronouns with explicit nouns** (reduce pronoun ratio)
2. **Flag high-surprisal tokens** (tokens the model finds unexpected)
3. **Report what's missing** (PQI breakdown)

This is a **rule-based diagnostic tool** — the precursor to a learnable rewriter.

In [ ]:
# ── CELL 22: Prompt Diagnostic Tool ───────────────────────────────────────────

def diagnose_prompt(text: str, verbose: bool = True) -> dict:
    """Full diagnostic report on a prompt's linguistic quality."""
    pos  = analyse_pos(text)
    surp = score_surprisal(text)
    sem  = score_semantic_density(text)

    issues = []
    suggestions = []

    if pos.pronoun_ratio > 0.10:
        issues.append(f"High pronoun ratio ({pos.pronoun_ratio:.0%}) — introduces reference ambiguity")
        suggestions.append("Replace pronouns (it/they/this/that) with explicit nouns or named entities")

    if pos.lss < 1.5:
        issues.append(f"Low LSS ({pos.lss:.2f}) — too many function words relative to content words")
        suggestions.append("Add domain-specific nouns and verbs; remove filler phrases")

    if pos.ne_count == 0:
        issues.append("No named entities detected — topic is underspecified")
        suggestions.append("Name the specific concept, person, molecule, framework, or algorithm you mean")

    if surp.perplexity > 200:
        issues.append(f"High prompt perplexity ({surp.perplexity:.0f}) — model finds this prompt unusual")
        suggestions.append(f"Most surprising tokens: {surp.high_surprisal_tokens} — consider rephrasing these")

    if sem.semantic_density < 0.3:
        issues.append(f"Low semantic density ({sem.semantic_density:.3f}) — content words are semantically scattered")
        suggestions.append("Keep the prompt focused on a single coherent topic")

    # PQI estimate (without sklearn, just a simple weighted sum)
    pqi_raw = (pos.lss * 0.35) + (sem.semantic_density * 0.25) + (pos.ne_count * 0.15) \
              - (pos.pronoun_ratio * 10 * 0.15) - (min(surp.perplexity, 500) / 500 * 0.10)

    if verbose:
        print("=" * 60)
        print(f"PROMPT: {text[:80]}..." if len(text) > 80 else f"PROMPT: {text}")
        print("=" * 60)
        print(f"  LSS             : {pos.lss:.3f}")
        print(f"  Pronoun ratio   : {pos.pronoun_ratio:.1%}")
        print(f"  Named entities  : {pos.ne_count}")
        print(f"  GPT-2 Perplexity: {surp.perplexity:.1f}")
        print(f"  Semantic density: {sem.semantic_density:.3f}")
        print(f"  PQI estimate    : {pqi_raw:.3f}")
        print()
        if issues:
            print("  ISSUES FOUND:")
            for i, (issue, sug) in enumerate(zip(issues, suggestions), 1):
                print(f"  [{i}] {issue}")
                print(f"      → {sug}")
        else:
            print("  No major issues detected.")
        print()

    return {"lss": pos.lss, "pronoun_ratio": pos.pronoun_ratio,
            "ne_count": pos.ne_count, "perplexity": surp.perplexity,
            "semantic_density": sem.semantic_density, "pqi": pqi_raw,
            "issues": issues, "suggestions": suggestions}


# ── Test it on a vague prompt
_ = diagnose_prompt("Can you help me understand what it does and how it affects things?")

In [ ]:
# ── CELL 23: Try the diagnostic on YOUR OWN prompts ───────────────────────────
# Edit the list below and re-run to get instant feedback.

MY_PROMPTS = [
    "Summarize the document and extract the key points from it.",
    "Extract the top 5 risk factors identified in the Q3 2024 board report for Acme Corp, ranked by financial impact.",
    "What does this function do and why does it fail sometimes?",
    "Identify the off-by-one error in the Python binary search implementation that causes IndexError on empty lists.",
]

for p in MY_PROMPTS:
    diagnose_prompt(p, verbose=True)

---
## Results & Interpretation

### What the data should show

| Feature | Hypothesis | Expected direction |
|---|---|---|
| LSS | Specific prompts have more content words | specific > vague |
| Pronoun Ratio | Vague prompts rely on pronouns | vague > specific |
| Named Entity Count | Specific prompts name things | specific > vague |
| GPT-2 Perplexity | Specific prompts match training distribution better | vague > specific |
| Semantic Density | Specific prompts cluster around one topic | specific > vague |
| PQI | Composite score separates classes | specific > vague |

### What this enables next

1. **With LLM labels** (`RUN_LLM=True`): fit logistic regression on `is_specific` → learn feature weights for PQI
2. **Scale up**: run on TruthfulQA / MMLU / HotpotQA prompts, correlate PQI with exact-match accuracy  
3. **Entropy-gated decoding**: use the surprisal profiler *during generation* to detect uncertainty spikes → apply dynamic logit bias
4. **Automated rewriter**: fine-tune a small model to rewrite low-PQI prompts into high-PQI prompts

---
*Research notebook — mycontext/research/hypothesis_1_pos_entropy_experiment.ipynb*